# Assignment 2 — Reward Model (TRL + LoRA + HH-RLHF)

This notebook trains a **Reward Model (RM)** using **Hugging Face TRL** and **PEFT (LoRA)** on the **Anthropic HH-RLHF** preference dataset.

This version is made for **Google Colab**. It mounts **Google Drive** and saves the model and result files there.

What you get:
- A reproducible reward-model training run
- A small reward model built from a causal LM backbone plus a scalar reward head
- Saved model/adapters to disk
- Pairwise reward scores on fixed evaluation pairs, saved to `results/rm_pair_scores.json`

Docs:
- TRL RewardTrainer docs: https://huggingface.co/docs/trl/en/reward_trainer
- HH-RLHF dataset: https://huggingface.co/datasets/Anthropic/hh-rlhf

> Recommended small backbone: `Qwen/Qwen2.5-0.5B`.


## Step 1: Install packages

Run this cell in Colab first. If Colab asks, restart the runtime after install.


In [ ]:
# Install packages for Colab.
# If Colab says restart runtime, do that and then run the notebook again.
!pip -q install -U "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
                  "trl==0.29.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "sentencepiece" "wandb" "evaluate"

## Step 2: Mount Google Drive

This step connects Colab to your Google Drive. The trained model and result files will be saved there.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive is ready.')
except ImportError:
    print('This notebook is not running in Google Colab.')


## Step 3: Import and set basic options

This part imports the tools, sets the random seed, chooses the model and dataset, and creates folders in Google Drive.


In [ ]:
import os, json, random
from datetime import datetime

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
from peft import LoraConfig
from trl import RewardConfig, RewardTrainer

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# -----------------------
# Model choice
# -----------------------
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
# MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # larger alternative if you have more VRAM

# -----------------------
# Dataset choice
# -----------------------
RM_DATASET = "Anthropic/hh-rlhf"

# Keep this notebook practical for Colab.
# FAST_MODE is tuned for a T4-level Colab runtime and aims for roughly 20 minutes.
FAST_MODE = True

if FAST_MODE:
    MAX_TRAIN_SAMPLES = 1536
    MAX_EVAL_SAMPLES = 256
    MAX_LENGTH = 256
    GRADIENT_ACCUMULATION_STEPS = 8
    WARMUP_STEPS = 10
    LOGGING_STEPS = 10
    EVAL_STRATEGY = "no"
    SAVE_STRATEGY = "epoch"
else:
    MAX_TRAIN_SAMPLES = 12000
    MAX_EVAL_SAMPLES = 1000
    MAX_LENGTH = 512
    GRADIENT_ACCUMULATION_STEPS = 16
    WARMUP_STEPS = 30
    LOGGING_STEPS = 20
    EVAL_STRATEGY = "epoch"
    SAVE_STRATEGY = "epoch"

SAMPLE_PAIRS_TO_SAVE = 20

# -----------------------
# Paths
# -----------------------
IN_COLAB = "COLAB_GPU" in os.environ
RUN_NAME = "reward_model_hh_rlhf_lora"
DRIVE_ROOT = "/content/drive/MyDrive/RLHF4LLMs"

if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    raise ValueError("Please run the Google Drive mount cell first.")

BASE_DIR = os.path.join(DRIVE_ROOT, RUN_NAME) if IN_COLAB else os.path.abspath(RUN_NAME)
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(BASE_DIR, "final_model")
TB_LOG_DIR = os.path.join(BASE_DIR, "tb_logs")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

for path in [BASE_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, TB_LOG_DIR, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Final model:", FINAL_MODEL_DIR)
print("TensorBoard logs:", TB_LOG_DIR)
print("Results:", RESULTS_DIR)
print("Model:", MODEL_NAME)
print("Dataset:", RM_DATASET)
print("FAST_MODE:", FAST_MODE)

## Step 4: Load and prepare HH-RLHF

HH-RLHF gives preference pairs with `chosen` and `rejected` responses. We clean the text, drop bad rows, keep a smaller sample for Colab, and save dataset info for the report.

> Note: this dataset contains harmful, unsafe, and offensive content by design.


In [ ]:
raw_train = load_dataset(RM_DATASET, split="train")
raw_eval = load_dataset(RM_DATASET, split="test")

def normalize_pair(example):
    return {
        "chosen": example["chosen"].strip(),
        "rejected": example["rejected"].strip(),
    }

train_ds = raw_train.map(normalize_pair, remove_columns=raw_train.column_names)
eval_ds = raw_eval.map(normalize_pair, remove_columns=raw_eval.column_names)

train_ds = train_ds.filter(lambda x: x["chosen"] != "" and x["rejected"] != "" and x["chosen"] != x["rejected"])
eval_ds = eval_ds.filter(lambda x: x["chosen"] != "" and x["rejected"] != "" and x["chosen"] != x["rejected"])

if MAX_TRAIN_SAMPLES is not None:
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))
if MAX_EVAL_SAMPLES is not None:
    eval_ds = eval_ds.shuffle(seed=SEED).select(range(min(MAX_EVAL_SAMPLES, len(eval_ds))))

dataset_info = {
    "dataset_name": RM_DATASET,
    "train_rows": len(train_ds),
    "eval_rows": len(eval_ds),
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_eval_samples": MAX_EVAL_SAMPLES,
    "example": train_ds[0],
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "dataset_info.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))
train_ds[0]

## Step 5: Load tokenizer and reward model backbone

A reward model is a sequence-classification model that outputs one scalar reward for a full response sequence.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

print(type(model).__name__)
print("Pad token id:", model.config.pad_token_id)

## Step 6: Add LoRA

We fine-tune a small adapter instead of all model weights. The `score` layer is saved because it is the new scalar reward head.


In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["score"],
)

## Step 7: Set training options

This part builds the reward trainer. The settings below are conservative so the notebook is more likely to fit on a small Colab GPU.


In [ ]:
reward_args = RewardConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=1e-4,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    eval_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=2,
    max_length=MAX_LENGTH,
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    logging_dir=TB_LOG_DIR,
    report_to=["tensorboard"],
    run_name=RUN_NAME,
    center_rewards_coefficient=0.01,
)

trainer = RewardTrainer(
    model=model,
    args=reward_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

## Step 8: Optional TensorBoard view

You can run this in Colab to watch training logs.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {TB_LOG_DIR}

## Step 9: Train the reward model

If training stops, this cell can resume from the latest checkpoint.


In [ ]:
MANUAL_RESUME_CHECKPOINT = None

def find_latest_checkpoint(checkpoint_dir):
    if not os.path.isdir(checkpoint_dir):
        return None
    checkpoint_names = [name for name in os.listdir(checkpoint_dir) if name.startswith("checkpoint-")]
    if not checkpoint_names:
        return None
    checkpoint_names = sorted(checkpoint_names, key=lambda x: int(x.split("-")[-1]))
    return os.path.join(checkpoint_dir, checkpoint_names[-1])

resume_checkpoint = None
if MANUAL_RESUME_CHECKPOINT and os.path.isdir(MANUAL_RESUME_CHECKPOINT):
    resume_checkpoint = MANUAL_RESUME_CHECKPOINT
else:
    resume_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR)

if resume_checkpoint:
    print("Resuming training from:", resume_checkpoint)
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
else:
    print("Starting training from scratch.")
    train_result = trainer.train()

train_metrics = dict(train_result.metrics)
train_metrics["model_name"] = MODEL_NAME
train_metrics["dataset_name"] = RM_DATASET
train_metrics["resume_from_checkpoint"] = resume_checkpoint

metrics_path = os.path.join(RESULTS_DIR, "train_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, ensure_ascii=False, indent=2)

train_metrics

## Step 10: Evaluate pairwise preference accuracy

We score both `chosen` and `rejected` texts and measure how often the chosen text gets the higher reward.


In [ ]:
def score_text(model, tokenizer, text, max_length=MAX_LENGTH):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    return float(logits.squeeze().float().cpu().item())

trainer.model.eval()

eval_subset = eval_ds.select(range(min(200, len(eval_ds))))
correct = 0
scored_pairs = []

for example in eval_subset:
    chosen_reward = score_text(trainer.model, tokenizer, example["chosen"])
    rejected_reward = score_text(trainer.model, tokenizer, example["rejected"])
    preferred_correct = chosen_reward > rejected_reward
    correct += int(preferred_correct)
    scored_pairs.append({
        "chosen_reward": chosen_reward,
        "rejected_reward": rejected_reward,
        "preferred_correct": preferred_correct,
    })

pairwise_accuracy = correct / len(scored_pairs)
eval_metrics = {
    "pairwise_accuracy": pairwise_accuracy,
    "num_pairs": len(scored_pairs),
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "eval_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, ensure_ascii=False, indent=2)

eval_metrics

## Step 11: Save the trained reward model

This saves the final model, tokenizer, and basic run metadata.


In [ ]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

save_info = {
    "base_dir": BASE_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "final_model_dir": FINAL_MODEL_DIR,
    "results_dir": RESULTS_DIR,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "save_info.json"), "w", encoding="utf-8") as f:
    json.dump(save_info, f, ensure_ascii=False, indent=2)

print("Saved final model to:", FINAL_MODEL_DIR)
print("Saved result files to:", RESULTS_DIR)

## Step 12: Save fixed pair scores

We save a few evaluation pairs with both rewards so you can inspect whether the model usually prefers the better answer.


In [ ]:
sample_pairs = eval_ds.select(range(min(SAMPLE_PAIRS_TO_SAVE, len(eval_ds))))
rm_pair_scores = []

for i, example in enumerate(sample_pairs):
    chosen_reward = score_text(trainer.model, tokenizer, example["chosen"])
    rejected_reward = score_text(trainer.model, tokenizer, example["rejected"])
    rm_pair_scores.append({
        "pair_id": i,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
        "chosen_reward": chosen_reward,
        "rejected_reward": rejected_reward,
        "preferred_correct": chosen_reward > rejected_reward,
    })

pair_scores_path = os.path.join(RESULTS_DIR, "rm_pair_scores.json")
with open(pair_scores_path, "w", encoding="utf-8") as f:
    json.dump(rm_pair_scores, f, ensure_ascii=False, indent=2)

print("Pair scores saved to:", pair_scores_path)
rm_pair_scores[0]